# Two-sample KS analysis of deviated angles

This notebook reproduces the same two-sample Kolmogorov-Smirnov procedure used for the paper analysis, using a public synthetic example. The example values are not experimental data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

DATA_FILE = Path("../outputs/ks_deviated_angles/example_synthetic_deviated_angles.xlsx")
SHEET_NAME = "Deviated angles"
ALPHA = 0.05

# Edit this list to select any comparisons present in the workbook.
COMPARISONS = [
    ("CONTROL", "TREATMENT_W"),
    ("CONTROL", "TREATMENT_X"),
    ("CONTROL", "TREATMENT_Y"),
    ("CONTROL", "TREATMENT_Z"),
]


In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE.resolve()}\n"
        "Run this notebook from the notebooks directory or update DATA_FILE."
    )

data = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)
data.columns = data.columns.str.strip()
print("Available conditions:", data.columns.tolist())
data.head()


In [ ]:
requested = {condition for pair in COMPARISONS for condition in pair}
missing = sorted(requested.difference(data.columns))
if missing:
    raise KeyError(f"Conditions not found in the workbook: {missing}")

rows = []
for first, second in COMPARISONS:
    first_values = pd.to_numeric(data[first], errors="coerce").dropna()
    second_values = pd.to_numeric(data[second], errors="coerce").dropna()
    statistic, p_value = ks_2samp(first_values, second_values, alternative="two-sided", method="auto")
    rows.append({
        "First condition": first,
        "Second condition": second,
        "n first": len(first_values),
        "n second": len(second_values),
        "KS statistic": statistic,
        "p value": p_value,
        f"Significant (alpha={ALPHA})": p_value < ALPHA,
    })

results = pd.DataFrame(rows)
results


## Changing the comparisons

Edit only `COMPARISONS` in the first code cell. Each item must contain two column names exactly as they appear in the Excel file. For example:

```python
COMPARISONS = [
    ("TREATMENT_W", "TREATMENT_X"),
    ("CONTROL", "TREATMENT_Z"),
]
```

The test is two-sided. A result with `p value < ALPHA` is labelled significant. For confirmatory analyses with many comparisons, consider a prespecified multiple-testing correction and report it explicitly.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for condition in sorted(requested):
    values = pd.to_numeric(data[condition], errors="coerce").dropna().sort_values()
    cumulative = range(1, len(values) + 1)
    ax.step(values, [value / len(values) for value in cumulative], where="post", label=condition)
ax.set(xlabel="Deviated angle (degrees)", ylabel="Empirical cumulative probability")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
